## Classificazione e Clustering Pokemon
### Classificazione
#### Cella 1: Importazioni e Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense # type: ignore
from tensorflow.keras.models import Model, Sequential # type: ignore
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4" # in order to avoid memory error

In [ ]:
# Carica il dataset
df = pd.read_csv("pokemon.csv")

In [ ]:
# Preprocessing: elimina duplicati e gestisci i valori mancanti
def preprocess_data(df):
    # Elimina i duplicati
    df.drop_duplicates(keep='first', inplace=True)
    
    # Seleziona le colonne numeriche
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    # Sostituisci i valori mancanti con la mediana
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    
    # Seleziona le colonne categoriche
    categorical_cols = df.select_dtypes(include=['object']).columns
    # Sostituisci i valori mancanti con "Unknown"
    df[categorical_cols] = df[categorical_cols].fillna("Unknown")
    
    # Converti le colonne stringa in liste Python
    df['type'] = df['type'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
    df['charged_moves'] = df['charged_moves'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
    df['fast_moves'] = df['fast_moves'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
    
    # Combina le mosse in un'unica lista
    df['all_moves'] = df['charged_moves'] + df['fast_moves']
    
    return df

df = preprocess_data(df)

In [ ]:
# Codifica le variabili categoriche
def encode_features(df):
    # Seleziona le feature numeriche
    numeric_features = ['base_attack', 'base_defense', 'base_stamina', 'max_cp',
                        'attack_probability', 'dodge_probability',
                        'max_pokemon_action_frequency', 'min_pokemon_action_frequency']
    X_numeric = df[numeric_features].values
    
    # Normalizza le feature numeriche
    scaler = StandardScaler()
    X_numeric_scaled = scaler.fit_transform(X_numeric)
    
    # Codifica la colonna "type" usando MultiLabelBinarizer
    mlb_type = MultiLabelBinarizer()
    X_type = mlb_type.fit_transform(df['type'])
    
    # Codifica le mosse
    mlb_moves = MultiLabelBinarizer()
    X_moves = mlb_moves.fit_transform(df['all_moves'])
    
    # Combina tutte le feature in un unico vettore
    X_combined = np.concatenate([X_numeric_scaled, X_type, X_moves], axis=1)
    
    return X_combined, numeric_features

X_combined, numeric_features = encode_features(df)

#### Cella 2: Costruzione e Addestramento dell'Autoencoder

In [ ]:
# Funzione per costruire e allenare l'autoencoder
def build_autoencoder(input_dim, encoding_dim=16):
    # Definizione dell'architettura dell'autoencoder
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(128, activation='relu')(input_layer)
    encoded = Dense(64, activation='relu')(encoded)
    latent = Dense(encoding_dim, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(latent)
    decoded = Dense(128, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    # Costruzione del modello
    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder

In [ ]:
# Costruzione e allenamento dell'autoencoder
autoencoder = build_autoencoder(X_combined.shape[1])
autoencoder.fit(X_combined, X_combined, epochs=50, batch_size=32, validation_split=0.2)

In [ ]:
# Estrazione della rappresentazione latente
encoder = Model(inputs=autoencoder.input, outputs=autoencoder.layers[2].output)
X_latent = encoder.predict(X_combined)

#### Cella 3: Analisi della Rappresentazione Latente

In [ ]:
# Funzione per visualizzare la distribuzione della rappresentazione latente
def plot_latent_distribution(X_latent):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=X_latent[:, 0], y=X_latent[:, 1], alpha=0.5)
    plt.xlabel("Latent Dim 1")
    plt.ylabel("Latent Dim 2")
    plt.title("Distribuzione della Rappresentazione Latente")
    plt.show()

plot_latent_distribution(X_latent)

#### Cella 4: Clustering con K-Means e Silhouette Score

In [ ]:
# Funzione per il clustering K-Means
def kmeans_clustering(X_latent, k_range=range(2, 11)):
    wcss = []  # Within-Cluster Sum of Squares
    silhouette_scores = []

    for k in k_range:  # Da 2 a 10 cluster
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_latent)
        wcss.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(X_latent, labels))
    
    return wcss, silhouette_scores

In [ ]:
# Eseguiamo il clustering
wcss, silhouette_scores = kmeans_clustering(X_latent)

In [ ]:
# Grafico del metodo del gomito
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(range(2, 11), wcss, marker='o', linestyle='--')
plt.xlabel("Numero di cluster (K)")
plt.ylabel("WCSS")
plt.title("Metodo del Gomito")

In [ ]:
# Grafico del silhouette score
plt.subplot(1, 2, 2)
plt.plot(range(2, 11), silhouette_scores, marker='s', linestyle='-')
plt.xlabel("Numero di cluster (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score per diversi K")
plt.show()

In [ ]:
# Applicare il numero ottimale di cluster (K=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_latent)

In [ ]:
# Aggiungiamo i cluster al dataset
df["cluster"] = clusters

In [ ]:
# Visualizza la distribuzione dei cluster
sns.countplot(x=df["cluster"], hue=df["cluster"], palette="viridis", legend=False)
plt.title("Distribuzione dei Cluster")
plt.xlabel("Cluster")
plt.ylabel("Conteggio")
plt.show()

#### Cella 5: Random Forest Classifier

In [ ]:
# Funzione per la classificazione con Random Forest
def random_forest_classification(X_train, y_train, X_test, y_test, features, target, label_encoder):
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    
    # Matrice di confusione
    conf_matrix = confusion_matrix(y_test, y_pred)
    sns.heatmap(conf_matrix, annot=True, cmap="Blues", fmt="d", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.xlabel("Predetto")
    plt.ylabel("Reale")
    plt.title("Matrice di Confusione")
    plt.show()
    
    # Report di classificazione
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, labels=label_encoder.transform(label_encoder.classes_), zero_division=1))
    
    return clf

In [ ]:
# Codifica il target 'rarity'
target = "rarity"
label_encoder = LabelEncoder()
df[target] = label_encoder.fit_transform(df[target])

In [ ]:
# Suddividi il dataset
X_train, X_test, y_train, y_test = train_test_split(df[numeric_features], df[target], test_size=0.2, random_state=42)

In [ ]:
# Esegui la classificazione
random_forest_classification(X_train, y_train, X_test, y_test, numeric_features, target, label_encoder)

#### Cella 6: Definizione del Target PvP vs. PvE

In [ ]:
# Funzione per assegnare lo stile PvP o PvE
def assign_pvp_pve_style(row):
    pvp_moves = {'Counter', 'Quick Attack', 'Vine Whip', 'Tackle', 'Bug Bite', 'Ember', 
                'Scratch', 'Water Gun', 'Bubble', 'Wing Attack', 'Peck'}
    pve_moves = {'Hydro Cannon', 'Blast Burn', 'Solar Beam', 'Flamethrower', 'Dragon Claw', 
                'Skull Bash', 'Ice Beam', 'Hydro Pump'}
    
    moves = set(row['all_moves'])
    if moves.intersection(pvp_moves) and not moves.intersection(pve_moves):
        return 0  # PvP
    else:
        return 1  # PvE

In [ ]:
# Aggiungi la colonna 'style'
df['style'] = df.apply(assign_pvp_pve_style, axis=1)

In [ ]:
# Suddividi in train e test per PvP vs PvE
X_train, X_test, y_train, y_test = train_test_split(X_latent, df['style'], test_size=0.2, random_state=42)

In [ ]:
# Classificazione PvP vs PvE con Random Forest
random_forest_classification(X_train, y_train, X_test, y_test, numeric_features, 'style', label_encoder)

### Clustering
#### Cella 1: Trovare il numero ottimale di cluster (Metodo del Gomito)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"  # Cambia il numero in base ai core del tuo PC

In [ ]:
# Testiamo diversi valori di K
wcss = []  # Within-Cluster Sum of Squares

for k in range(1, 11):  # Da 1 a 10 cluster
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_latent)
    wcss.append(kmeans.inertia_)  # Inertia = somma delle distanze dei punti dal centroide

In [ ]:
# Grafico del metodo del gomito
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.xlabel("Numero di cluster (K)")
plt.ylabel("WCSS (Within-Cluster Sum of Squares)")
plt.title("Metodo del Gomito per trovare il numero ottimale di cluster")
plt.show()

#### Cella 2: Applicare K-Means con il numero ottimale di cluster

In [ ]:
# Addestriamo K-Means con il numero ottimale di cluster (es. K=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_latent)

In [ ]:
# Aggiungiamo le etichette dei cluster al dataset
df["cluster"] = clusters

In [ ]:
# Visualizziamo la distribuzione dei cluster
print(df["cluster"].value_counts())

#### Cella 3: Visualizzare i cluster con PCA (2D)

In [ ]:
# Riduzione a 2 componenti principali
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_latent)

In [ ]:

# Creiamo un DataFrame con i dati ridotti e le etichette dei cluster
df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca["cluster"] = clusters

In [ ]:
# Grafico dei cluster
plt.figure(figsize=(8, 6))
for cluster in range(kmeans.n_clusters):
    subset = df_pca[df_pca["cluster"] == cluster]
    plt.scatter(subset["PC1"], subset["PC2"], label=f"Cluster {cluster}")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Visualizzazione dei cluster (PCA 2D)")
plt.legend()
plt.show()

####  Cella 4: KMeans per trovare cluster basati su feature latenti

In [ ]:
# Clustering per esplorare i dati
# KMeans per trovare cluster basati su feature latenti
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_combined)

In [ ]:
# Aggiungi i cluster al dataset
df["cluster"] = clusters

In [ ]:
# Visualizza la distribuzione dei cluster
sns.countplot(x=df["cluster"], hue=df["cluster"], palette="viridis", legend=False)
plt.title("Distribuzione dei Cluster")
plt.xlabel("Cluster")
plt.ylabel("Conteggio")
plt.show()

In [ ]:
# Addestramento di un classificatore Random Forest per PvP vs PvE
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
# Previsioni sul set di test
y_pred = clf.predict(X_test)

In [ ]:
# Matrice di confusione
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, cmap="Blues", fmt="d", xticklabels=["PvP", "PvE"], yticklabels=["PvP", "PvE"])
plt.xlabel("Predetto")
plt.ylabel("Reale")
plt.title("Matrice di Confusione")
plt.show()

In [ ]:
# Report di classificazione
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=["PvP", "PvE"]))